# EDA Notebook: Vehicle Price Modeling Data Quality

## tl;dr

This notebook profiles `CAR_DATA_OUTPUT/CAR_DATA_CLEANED.db` for current vehicle price modeling. It is designed for an 8 GB SQLite database, so default checks use deterministic rowid samples and indexed recent samples instead of full scans.

Initial sampled findings from a 50k listing sample:

- Prices are heavily right skewed: median about $26.8k, 95th percentile about $66.0k, 99th percentile about $128.9k, and max near $1.0M.
- The `>$150k` segment is sparse, about 0.77% of sampled listing rows, so it should be evaluated as a separate segment and not assumed to improve global MAE by default.
- The sampled cleaned listing rows have strong core completeness for VIN, price, mileage, and NHTSA make/model/year.
- Current-price modeling should default to one row per VIN, keeping the latest `loaddate`, so duplicate listings do not overweight repeatedly scraped vehicles.

The companion model script `ML/Price_ML_Models.py` now defaults to recent bounded sampling, latest-per-VIN deduplication, additional engineered features, and a candidate high-value classifier-routed LightGBM regressor.

## Context & Methods

### Modeling Goal

Predict the current listing price of a vehicle using scraped listing attributes, NHTSA VIN enrichment, and optional sentiment features. The intended row grain for the current-price model is one current listing observation per VIN.

### Key Assumptions

- `listings` contains repeated observations from recurring scrapes and multiple marketplaces.
- For current price prediction, repeated VIN rows are not independent training examples. The default modeling grain should be latest `loaddate` per VIN.
- Historical price/depreciation tasks can use repeated VIN observations, but validation must be time-aware and VIN-aware.
- `price_band` and `is_high_value` are labels or diagnostics. They should not be used as direct price-model features unless generated by a classifier at inference time.

### Literature Notes

Vehicle-price prediction commonly behaves like a hedonic pricing problem: price is explained by observed product attributes such as age, mileage, make/model, trim, fuel type, and market/source effects.

Relevant papers and methods:

- [How much is my car worth? Random Forest used-car pricing](https://arxiv.org/abs/1711.06970): a used-car price regression study using Random Forest after EDA-driven feature selection.
- [Vehicle Price Prediction by Aggregating Decision Tree Model With Boosting Model](https://arxiv.org/abs/2307.15982): emphasizes preprocessing and tree/boosting ensembles for vehicle price prediction.
- [ProbSAINT: Probabilistic Tabular Regression for Used Car Pricing](https://arxiv.org/abs/2403.03812): frames used-car pricing as tabular regression with uncertainty, useful for high-value or low-confidence predictions.
- [CatBoost: unbiased boosting with categorical features](https://arxiv.org/abs/1706.09516): relevant because this dataset has high-cardinality categorical fields like make/model/title/source/location.
- [NGBoost: Natural Gradient Boosting for Probabilistic Prediction](https://arxiv.org/abs/1910.03225): useful future option for price intervals rather than only point estimates.
- [AI Blue Book: Vehicle Price Prediction using Visual Features](https://arxiv.org/abs/1803.11227): shows vehicle images can add pricing signal if image URLs are later converted into visual embeddings or condition features.

In [1]:
from __future__ import annotations

import json
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

def find_workspace_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'CAR_DATA_OUTPUT' / 'CAR_DATA_CLEANED.db').exists():
            return candidate
    return start.resolve()

WORKSPACE = find_workspace_root(Path.cwd())
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

DB_PATH = WORKSPACE / 'CAR_DATA_OUTPUT' / 'CAR_DATA_CLEANED.db'
MODEL_SCRIPT = WORKSPACE / 'ML' / 'Price_ML_Models.py'
SAMPLE_SIZE = 50_000
MODEL_SAMPLE_SIZE = 50_000
RUN_FULL_SCAN = False
RUN_MODEL_BENCHMARK = False
HIGH_VALUE_THRESHOLD = 150_000

DB_PATH, DB_PATH.exists(), round(DB_PATH.stat().st_size / 1024**3, 2)

(WindowsPath('C:/Users/danie/Code/Car-Price-Data-Visualization-Learning/CAR_DATA_OUTPUT/CAR_DATA_CLEANED.db'),
 True,
 8.44)

## Data

The core data sources for current-price modeling are:

- `listings`: scraped listing rows, including VIN, price, mileage, source, location, title fields, and scrape/load dates.
- `nhtsa_enrichment`: VIN-decoded technical attributes and safety/recall/complaint enrichments.
- `price_history` and `listing_history`: repeated historical observations for depreciation or forecasting tasks.
- `vehicle_sentiment`: small model-year/make/model sentiment table from YouTube comments.

The acquisition code uses parallel browser workers against AutoTempest result sources, normalizes price/mileage, and stores source-specific rows. The strongest modeling risks are duplicate VIN observations, source mix drift, sparse high-value rows, and leakage from fields that are only known after the price outcome.

In [2]:
def fetch_schema_overview(db_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    with sqlite3.connect(db_path) as conn:
        tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
            """,
            conn,
        )
        overview_rows = []
        for table in tables['name']:
            count = conn.execute(f"SELECT COUNT(*) FROM '{table}'").fetchone()[0]
            columns = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
            overview_rows.append(
                {
                    'table': table,
                    'rows': count,
                    'columns': len(columns),
                    'column_names': ', '.join(row[1] for row in columns[:12]),
                }
            )
        indexes = pd.read_sql_query(
            """
            SELECT name, tbl_name, sql
            FROM sqlite_master
            WHERE type = 'index'
            ORDER BY tbl_name, name
            """,
            conn,
        )
    return pd.DataFrame(overview_rows), indexes

schema_overview, indexes = fetch_schema_overview(DB_PATH)
schema_overview

,table,rows,columns,column_names
0,listing_history,17914055,4,"vin, history_date, mileage, price"
1,listings,5732296,21,"vin, date, loaddate, title, location, location..."
2,nhtsa_enrichment,3348338,88,"vin, nhtsa_ABS, nhtsa_ActiveSafetySysNote, nht..."
3,price_history,12446036,5,"vin, history_date, mileage, price, trend"
4,sqlite_sequence,0,2,"name, seq"
5,vehicle_sentiment,622,6,"make, model, year, sentiment_score, comment_co..."


In [3]:
indexes

,name,tbl_name,sql
0,idx_listing_history_date,listing_history,CREATE INDEX idx_listing_history_date ON listi...
1,idx_listing_history_vin_date,listing_history,CREATE INDEX idx_listing_history_vin_date ON l...
2,sqlite_autoindex_listing_history_1,listing_history,None
3,idx_listings_loaddate,listings,CREATE INDEX idx_listings_loaddate ON listings...
4,idx_listings_price,listings,CREATE INDEX idx_listings_price ON listings (p...
5,idx_listings_vin,listings,CREATE INDEX idx_listings_vin ON listings (vin)
6,sqlite_autoindex_listings_1,listings,None
7,idx_nhtsa_make_model_year,nhtsa_enrichment,CREATE INDEX idx_nhtsa_make_model_year ON nhts...
8,sqlite_autoindex_nhtsa_enrichment_1,nhtsa_enrichment,None
9,idx_price_history_date,price_history,CREATE INDEX idx_price_history_date ON price_h...


### Deterministic Samples

The helper below samples evenly spaced rowids. This avoids expensive full-table scans while still spreading the sample across the database rather than taking only the first rows.

In [4]:
def read_even_rowid_sample(
    db_path: Path,
    table: str,
    sample_size: int,
    extra_select: str = '',
    extra_join: str = '',
) -> pd.DataFrame:
    with sqlite3.connect(db_path) as conn:
        max_rowid = conn.execute(f'SELECT MAX(rowid) FROM {table}').fetchone()[0]
        query = f"""
        WITH RECURSIVE seq(n) AS (
            VALUES(0)
            UNION ALL
            SELECT n + 1 FROM seq WHERE n < {sample_size - 1}
        ), sampled_ids AS (
            SELECT CAST(1 + n * (({max_rowid} - 1.0) / {sample_size}) AS INTEGER) AS rid
            FROM seq
        )
        SELECT base.* {extra_select}
        FROM sampled_ids AS sampled
        JOIN {table} AS base
            ON base.rowid = sampled.rid
        {extra_join}
        """
        return pd.read_sql_query(query, conn)

listing_extra_select = """,
    n.nhtsa_Make, n.nhtsa_Model, n.nhtsa_ModelYear, n.nhtsa_BodyClass,
    n.nhtsa_DriveType, n.nhtsa_FuelTypePrimary, n.nhtsa_TransmissionStyle,
    n.nhtsa_EngineHP, n.nhtsa_EngineCylinders, n.nhtsa_DisplacementL,
    n.nhtsa_total_recalls, n.nhtsa_total_complaints
"""
listing_extra_join = 'LEFT JOIN nhtsa_enrichment AS n USING(vin)'

listings_sample = read_even_rowid_sample(
    DB_PATH,
    'listings',
    SAMPLE_SIZE,
    extra_select=listing_extra_select,
    extra_join=listing_extra_join,
)
price_history_sample = read_even_rowid_sample(DB_PATH, 'price_history', SAMPLE_SIZE)

for frame in [listings_sample, price_history_sample]:
    for column in ['price', 'mileage']:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')

listings_sample.head()

,vin,date,loaddate,title,location,locationCode,countryCode,pendingSale,distance,priceRecentChange,price,mileage,sourceName,sellerType,listingType,vehicleTitle,vehicleTitleDesc,title_normalized,title_trim,trim_combined,trim_source,nhtsa_Make,nhtsa_Model,nhtsa_ModelYear,nhtsa_BodyClass,nhtsa_DriveType,nhtsa_FuelTypePrimary,nhtsa_TransmissionStyle,nhtsa_EngineHP,nhtsa_EngineCylinders,nhtsa_DisplacementL,nhtsa_total_recalls,nhtsa_total_complaints
0,JTEVB5BR8S5005439,2026-02-28,2026-03-05,2025 Toyota 4Runner TRD Off-Road HV,"Miami, FL","33,157.0000",US,0,0.0000,0,51590,5548,None,Dealer,regular,Clean,A clean title indicates that a vehicle has nev...,2025 TOYOTA 4RUNNER TRD OFF ROAD HV,TRD_OFF_ROAD,TRD_OFF_ROAD,title,TOYOTA,4RUNNER,2025,Sport Utility Vehicle (SUV)/Multi-Purpose Vehi...,4WD/4-Wheel Drive/4x4,Gasoline,Automatic,275.0000,4.0000,2.4000,1.0000,37.0000
1,3TMLB5JN8SM156768,2026-03-03,2026-03-05,2025 Toyota Tacoma Double Cab TRD Off-Road Pic...,"Jacksonville, FL","32,219.0000",US,0,327.2000,1,39907,13893,Carvana,Dealer,regular,Clean,A clean title indicates that a vehicle has nev...,2025 TOYOTA TACOMA DOUBLE CAB TRD OFF ROAD PIC...,TRD_OFF_ROAD,TRD_OFF_ROAD,title,TOYOTA,TACOMA,2025,Pickup,4WD/4-Wheel Drive/4x4,Gasoline,,278.0000,4.0000,2.4000,3.0000,37.0000
2,5TDKDRAHXPS045217,2026-03-04,2026-03-05,2023 Toyota Highlander L,"Miami, FL","33,130.0000",US,0,0.0000,0,32000,31099,Cars.com,Dealer,regular,Unknown,"This listing has an unknown title, please cont...",2023 TOYOTA HIGHLANDER L,,LE_L_LIMITED_PLATINUM_XLE_XSE_W_12_3_MULTIMEDI...,nhtsa_Trim,TOYOTA,HIGHLANDER,2023,Sport Utility Vehicle (SUV)/Multi-Purpose Vehi...,4x2,Gasoline,,265.0000,4.0000,2.4000,8.0000,135.0000
3,4T3LWRFV0SU167413,2026-02-20,2026-03-05,2025 Toyota RAV4 Hybrid LE Sport Utility 4D,"Haines City, FL","33,844.0000",US,0,183.2000,1,28856,30600,Carvana,Dealer,regular,Clean,A clean title indicates that a vehicle has nev...,2025 TOYOTA RAV4 HYBRID LE SPORT UTILITY 4D,SPORT,LE SPORT,nhtsa_Trim,TOYOTA,RAV4,2025,Sport Utility Vehicle (SUV)/Multi-Purpose Vehi...,4WD/4-Wheel Drive/4x4,Gasoline,Automatic,176.0000,4.0000,2.5000,2.0000,42.0000
4,JTDBCMFE5S3103040,2026-03-02,2026-03-05,2025 Toyota Corolla Hybrid LE,"Doral, FL","33,172.0000",US,0,10.1000,0,26014,5097,Cars.com,Dealer,regular,Clean,A clean title indicates that a vehicle has nev...,2025 TOYOTA COROLLA HYBRID LE,LE,LE_XLE_SE LE_W_CONVENIENCE_TECH_PKG_W_PREMIUM_...,nhtsa_Trim,TOYOTA,COROLLA,2025,Sedan/Saloon,4x2,Gasoline,,97.0000,4.0000,1.8000,NaN,17.0000


## Results

### Core Completeness and Shape

In [5]:
def quality_summary(df: pd.DataFrame, label: str) -> pd.DataFrame:
    output = {
        'dataset': label,
        'rows': len(df),
        'distinct_vins': df['vin'].nunique(dropna=True) if 'vin' in df else np.nan,
        'duplicate_vin_share': 1 - df['vin'].nunique(dropna=True) / len(df) if 'vin' in df and len(df) else np.nan,
        'missing_vin_rate': df['vin'].isna().mean() if 'vin' in df else np.nan,
        'null_price_rate': df['price'].isna().mean() if 'price' in df else np.nan,
        'nonpositive_price_rate': df['price'].le(0).fillna(False).mean() if 'price' in df else np.nan,
        'null_mileage_rate': df['mileage'].isna().mean() if 'mileage' in df else np.nan,
        'negative_mileage_rate': df['mileage'].lt(0).fillna(False).mean() if 'mileage' in df else np.nan,
    }
    if 'nhtsa_Make' in df:
        output.update(
            {
                'nhtsa_make_missing_rate': df['nhtsa_Make'].isna().mean(),
                'nhtsa_model_missing_rate': df['nhtsa_Model'].isna().mean(),
                'nhtsa_year_missing_rate': df['nhtsa_ModelYear'].isna().mean(),
            }
        )
    return pd.DataFrame([output])

pd.concat(
    [
        quality_summary(listings_sample, 'listings_sample'),
        quality_summary(price_history_sample, 'price_history_sample'),
    ],
    ignore_index=True,
)

,dataset,rows,distinct_vins,duplicate_vin_share,missing_vin_rate,null_price_rate,nonpositive_price_rate,null_mileage_rate,negative_mileage_rate,nhtsa_make_missing_rate,nhtsa_model_missing_rate,nhtsa_year_missing_rate
0,listings_sample,50000,49682,0.0064,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,price_history_sample,50000,49767,0.0047,0.0000,0.0000,0.0000,0.0552,0.0000,NaN,NaN,NaN


In [6]:
price_quantiles = listings_sample.loc[listings_sample['price'].gt(0), 'price'].quantile(
    [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
)
mileage_quantiles = listings_sample.loc[listings_sample['mileage'].ge(0), 'mileage'].quantile(
    [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
)

pd.DataFrame({'price': price_quantiles, 'mileage': mileage_quantiles})

,price,mileage
0.0000,"1,000.0000",1.0000
0.0100,"5,495.0000",5.0000
0.0500,"9,500.0000","1,355.7000"
0.1000,"12,988.9000","6,349.1000"
0.2500,"19,175.5000","21,214.0000"
0.5000,"26,990.0000","45,086.0000"
0.7500,"37,944.5000","81,610.5000"
0.9000,"52,500.0000","117,382.6000"
0.9500,"66,861.4500","141,429.4500"
0.9900,"133,025.7600","196,387.0700"


In [7]:
price_band = pd.cut(
    listings_sample['price'],
    bins=[-np.inf, 0, 25_000, 50_000, 100_000, HIGH_VALUE_THRESHOLD, np.inf],
    labels=['nonpositive_or_missing', 'under_25k', '25k_50k', '50k_100k', '100k_150k', '150k_plus'],
)
price_band_summary = (
    price_band.value_counts(dropna=False)
    .sort_index()
    .rename_axis('price_band')
    .reset_index(name='rows')
)
price_band_summary['share'] = price_band_summary['rows'] / price_band_summary['rows'].sum()
price_band_summary

,price_band,rows,share
0,nonpositive_or_missing,0,0.0000
1,under_25k,22302,0.4460
2,25k_50k,22020,0.4404
3,50k_100k,4870,0.0974
4,100k_150k,388,0.0078
5,150k_plus,420,0.0084


In [8]:
fig = px.histogram(
    listings_sample[listings_sample['price'].between(1, 250_000)],
    x='price',
    nbins=80,
    title='Listing Price Distribution Capped at $250k for Readability',
)
fig.update_layout(xaxis_title='price', yaxis_title='sampled rows')
fig.show()

In [9]:
fig = px.bar(price_band_summary, x='price_band', y='share', title='Sampled Price Band Share')
fig.update_layout(xaxis_title='price band', yaxis_title='share of sampled rows')
fig.show()

### Duplicate VIN Handling

For the current-price model, duplicate VIN rows can overweight vehicles that were scraped repeatedly or listed across sources. The recommended default is latest `loaddate` per VIN. Historical/depreciation models should keep repeated observations, but they need time-aware validation and should avoid future leakage.

In [10]:
def latest_per_vin(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    working = df.copy()
    for column in ['loaddate', 'date']:
        if column in working:
            working[column] = pd.to_datetime(working[column], errors='coerce')
    sort_columns = [column for column in ['vin', 'loaddate', 'date'] if column in working]
    if sort_columns:
        working = working.sort_values(sort_columns)
    deduped = working.drop_duplicates('vin', keep='last').reset_index(drop=True)
    summary = pd.DataFrame(
        [
            {
                'rows_before': len(working),
                'rows_after': len(deduped),
                'distinct_vins_before': working['vin'].nunique(dropna=True),
                'rows_removed': len(working) - len(deduped),
                'removed_share': (len(working) - len(deduped)) / len(working),
            }
        ]
    )
    return deduped, summary

latest_listings_sample, dedup_summary = latest_per_vin(listings_sample)
dedup_summary

,rows_before,rows_after,distinct_vins_before,rows_removed,removed_share
0,50000,49682,49682,318,0.0064


In [11]:
source_quality = (
    latest_listings_sample.assign(
        bad_price=latest_listings_sample['price'].isna() | latest_listings_sample['price'].le(0),
        bad_mileage=latest_listings_sample['mileage'].isna() | latest_listings_sample['mileage'].lt(0),
        high_value=latest_listings_sample['price'].gt(HIGH_VALUE_THRESHOLD),
    )
    .groupby(latest_listings_sample['sourceName'].fillna('MISSING'), dropna=False)
    .agg(
        rows=('vin', 'size'),
        vins=('vin', 'nunique'),
        bad_price_rows=('bad_price', 'sum'),
        bad_mileage_rows=('bad_mileage', 'sum'),
        avg_price=('price', 'mean'),
        high_value_rows=('high_value', 'sum'),
    )
    .sort_values('rows', ascending=False)
    .head(20)
    .reset_index(names='sourceName')
)
source_quality

,sourceName,rows,vins,bad_price_rows,bad_mileage_rows,avg_price,high_value_rows
0,Cars.com,32334,32334,0,0,"33,067.9282",360
1,TrueCar,11990,11990,0,0,"30,002.6370",31
2,Carvana,2218,2218,0,0,"27,850.7863",0
3,CarGurus,1350,1350,0,0,"42,281.8185",3
4,eBay,809,809,0,0,"32,590.5562",13
5,CarSoup,594,594,0,0,"44,319.3300",0
6,AutoTempest,184,184,0,0,"38,468.8641",0
7,PrivateAuto,68,68,0,0,"28,801.2647",0
8,Hemmings Classifieds,60,60,0,0,"47,162.8333",4
9,AutoWeb,27,27,0,0,"27,316.5556",0


In [12]:
make_mix = (
    latest_listings_sample.assign(high_value=latest_listings_sample['price'].gt(HIGH_VALUE_THRESHOLD))
    .groupby(latest_listings_sample['nhtsa_Make'].fillna('MISSING'), dropna=False)
    .agg(
        rows=('vin', 'size'),
        vins=('vin', 'nunique'),
        avg_price=('price', 'mean'),
        high_value_rows=('high_value', 'sum'),
    )
    .sort_values('rows', ascending=False)
    .head(25)
    .reset_index(names='make')
)
make_mix

,make,rows,vins,avg_price,high_value_rows
0,TOYOTA,4219,4219,"29,733.9618",0
1,FORD,3969,3969,"33,373.6357",2
2,HONDA,3908,3908,"25,174.4084",0
3,CHEVROLET,3496,3496,"31,345.7120",1
4,NISSAN,3471,3471,"21,417.8585",4
5,JEEP,3420,3420,"27,329.1167",0
6,GMC,2376,2376,"39,905.1368",0
7,RAM,2169,2169,"38,610.3250",1
8,SUBARU,2124,2124,"24,069.3277",0
9,KIA,2052,2052,"23,291.6301",0


### Feature Engineering Recommendations

Recommended current-price features:

- Vehicle identity: make, model, model year, make-model-year, body class, trim/title tokens, source, seller/listing type.
- Depreciation shape: vehicle age, age squared, mileage, log mileage, mileage per year, mileage bucket, mileage-age interaction.
- Technical attributes: fuel type, electrification, drive type, transmission, engine cylinders, horsepower, displacement, doors, seats, recall/complaint counts.
- Market context: source name, location region, listing recency, listing month/week, pending-sale and price-recent-change flags.
- Text signals: title length, word count, certified/CPO token, AWD/4WD token, luxury/performance trim token.
- Segment diagnostics: price bands and `>$150k` labels for evaluation; do not pass actual price band as a raw feature to the final regressor.

Modeling recommendation:

1. Use a leakage-safe baseline: latest row per VIN, time-aware split when enough dates exist, and zero VIN overlap between train/test.
2. Compare global log-price regressors against a high-value routed model.
3. Select by global MAE/RMSLE and by segment MAE for `>$150k`, not just one aggregate metric.
4. Consider CatBoost later because the dataset is categorical-heavy; consider probabilistic models such as NGBoost or ProbSAINT-style uncertainty when pricing rare or exotic vehicles.

In [13]:
if RUN_MODEL_BENCHMARK:
    from ML.Price_ML_Models import train_current_price_models

    report = train_current_price_models(
        db_path=DB_PATH,
        sample_size=MODEL_SAMPLE_SIZE,
        sample_strategy='recent',
        deduplicate_vins=True,
    )
    print(json.dumps({
        'recommended_model': report['recommended_model'],
        'row_counts': report['row_counts'],
        'deduplication': report['deduplication'],
    }, indent=2))
else:
    print('Set RUN_MODEL_BENCHMARK = True to train the current-price model benchmark from this notebook.')

Set RUN_MODEL_BENCHMARK = True to train the current-price model benchmark from this notebook.


### Optional Full-Scan Checks

The default notebook avoids full scans. If you want exact rates, set `RUN_FULL_SCAN = True`. Expect these queries to take much longer on the 8 GB SQLite database.

In [14]:
if RUN_FULL_SCAN:
    listings_price_bands_query = f"""
        SELECT
            CASE
                WHEN price IS NULL THEN 'null'
                WHEN price <= 0 THEN 'nonpositive'
                WHEN price <= 25000 THEN 'under_25k'
                WHEN price <= 50000 THEN '25k_50k'
                WHEN price <= 100000 THEN '50k_100k'
                WHEN price <= {HIGH_VALUE_THRESHOLD} THEN '100k_150k'
                ELSE '150k_plus'
            END AS price_band,
            COUNT(*) AS rows,
            COUNT(DISTINCT vin) AS vins
        FROM listings
        GROUP BY price_band
        ORDER BY rows DESC
    """
    latest_vin_counts_query = """
        SELECT COUNT(*) AS latest_rows
        FROM (
            SELECT vin, MAX(loaddate) AS latest_loaddate
            FROM listings
            WHERE vin IS NOT NULL
            GROUP BY vin
        )
    """
    with sqlite3.connect(DB_PATH) as conn:
        display(pd.read_sql_query(listings_price_bands_query, conn))
        display(pd.read_sql_query(latest_vin_counts_query, conn))
else:
    print('Full-scan checks skipped. Set RUN_FULL_SCAN = True to run them.')

Full-scan checks skipped. Set RUN_FULL_SCAN = True to run them.


## Takeaways

- Use latest-per-VIN current listing rows for the current-price model. Repeated VINs are useful for depreciation/history tasks, but they are not independent current-price examples.
- Keep the target on a log scale for most regressors because the price target is severely right skewed.
- Evaluate the `>$150k` slice separately. A classifier-routed regressor is worth testing because the high-value segment is rare and source/make composition differs from everyday vehicles.
- Prefer indexed recent sampling for model iteration and deterministic rowid sampling for broad EDA. Reserve full-table scans for final validation or overnight runs.
- The acquisition pipeline is collecting useful high-signal fields, but source mix and repeated VINs must be measured in every model report.